# Data Preparation and Cleaning

This notebook prepares the movie and book datasets for our capstone project: a generative
recommendation system that focuses on the **cold-start problem**.

Cold start happens when the system has little or no interaction history to learn from.

For our project this can mean:

- **New users** who have few or no ratings yet. This includes collaborative filtering that has nothing to go on.
- **New movies or books** that have few or no ratings. They never show up in similarity-based recommendations.
- In these cases the model has to fall back on **content information** (genres, descriptions,
  authors, directors, release years, etc.) to make reasonable recommendations.

The cleaning pipeline in this notebook will:

1. Load and inspect the raw movie, book, and interaction data.
2. Clean and standardize each dataset.
3. Build a unified item table (movies + books) with a combined text feature for content-based / generative models.
4. Create train / validation / test splits, plus dedicated **new-user** and **new-item** cold-start test sets.
5. Save everything as CSV files for the modeling team.


### Import libraries

In [1]:
# Library
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
np.random.seed(SEED)

# make pandas output a bit easier to read
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

ModuleNotFoundError: No module named 'matplotlib'

### Load the datasets

We will be evaluating Netflix's Movie and Book dataset and will load the following three files:

- the **movie** dataset (Netflix-related, ~2023)
- the **book** dataset
- the **user–item interaction / ratings** data (if stored separately)

**Replace the file paths below with the actual locations of the files.**

In [ ]:
# TODO: replace these paths with the real file locations
movie_path = "data/movies.csv"
book_path = "data/books.csv"
ratings_path = "data/ratings.csv"   # set to None if interactions are inside the item files

In [ ]:
# load the movie dataset
movies_raw = pd.read_csv(movie_path)
print("Movies loaded:", movies_raw.shape)

In [ ]:
# load the book dataset
books_raw = pd.read_csv(book_path)
print("Books loaded:", books_raw.shape)

In [ ]:
# load the ratings / interactions dataset (skip if it does not exist as a separate file)
ratings_raw = None
if ratings_path is not None and os.path.exists(ratings_path):
    ratings_raw = pd.read_csv(ratings_path)
    print("Ratings loaded:", ratings_raw.shape)
else:
    print("No separate ratings file found - update ratings_path if this is wrong.")

### Inspect data

In [ ]:
def inspect(df, name):
    # quick overview of one dataframe, for all three datasets
    print("=" * 60)
    print(name)
    print("=" * 60)
    print("Shape:", df.shape)
    print("\nFirst 5 rows:")
    display(df.head())
    print("Columns:", list(df.columns))
    print("\nData types:")
    print(df.dtypes)
    print("\nDuplicate rows:", df.duplicated().sum())
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    print("\nMissing values:")
    print(pd.DataFrame({"missing": missing, "missing_%": missing_pct}))

inspect(movies_raw, "MOVIES")

In [ ]:
inspect(books_raw, "BOOKS")

In [ ]:
if ratings_raw is not None:
    inspect(ratings_raw, "RATINGS / INTERACTIONS")

In [ ]:
# small summary table of the raw datasets
summary_rows = [
    {"dataset": "movies", "rows": movies_raw.shape[0], "columns": movies_raw.shape[1],
     "duplicate_rows": movies_raw.duplicated().sum()},
    {"dataset": "books", "rows": books_raw.shape[0], "columns": books_raw.shape[1],
     "duplicate_rows": books_raw.duplicated().sum()},
]
if ratings_raw is not None:
    summary_rows.append({"dataset": "ratings", "rows": ratings_raw.shape[0],
                         "columns": ratings_raw.shape[1],
                         "duplicate_rows": ratings_raw.duplicated().sum()})

pd.DataFrame(summary_rows)

**Observations (edit after running):**

- Movies: *e.g., X rows, Y columns, notable missing values in ...*
- Books: *...*
- Ratings: *...*


## 5. Standardize column names

Lowercase, strip spaces, and replace spaces/punctuation with underscores so the rest of
the notebook can use consistent names.

In [ ]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ \-/]+", "_", regex=True)
        .str.replace(r"[^0-9a-z_]", "", regex=True)
    )
    return df

movies = clean_columns(movies_raw)
books = clean_columns(books_raw)
ratings = clean_columns(ratings_raw) if ratings_raw is not None else None

print("Movie columns: ", list(movies.columns))
print("Book columns:  ", list(books.columns))
if ratings is not None:
    print("Rating columns:", list(ratings.columns))

## 6. Identify important columns

The exact column names depend on the files we end up using, so instead of guessing
silently, we set them as variables in one place. **Update these after checking the
printed column lists above.**

In [ ]:
# ---- MOVIE dataset columns (update to match your file) ----
movie_id_col = "show_id"        # unique movie identifier
movie_title_col = "title"
movie_genre_col = "listed_in"   # Netflix datasets often call genres "listed_in"
movie_desc_col = "description"
movie_director_col = "director"
movie_cast_col = "cast"
movie_year_col = "release_year"

# ---- BOOK dataset columns (update to match your file) ----
book_id_col = "isbn"            # or "book_id"
book_title_col = "title"
book_author_col = "author"      # or "authors"
book_genre_col = "genre"        # or "categories"
book_desc_col = "description"
book_year_col = "publication_year"

# ---- RATINGS dataset columns (update to match your file) ----
user_col = "user_id"
rating_item_col = "item_id"     # the raw item id in the ratings file
rating_col = "rating"
timestamp_col = "timestamp"     # set to None if there is no timestamp

In [ ]:
# simple existence checks so wrong names fail loudly, not silently
def check_cols(df, cols, name):
    for c in cols:
        if c is not None and c not in df.columns:
            print(f"WARNING: '{c}' not found in {name} columns -> update the variable above")

check_cols(movies, [movie_id_col, movie_title_col, movie_genre_col, movie_desc_col,
                    movie_director_col, movie_cast_col, movie_year_col], "movies")
check_cols(books, [book_id_col, book_title_col, book_author_col, book_genre_col,
                   book_desc_col, book_year_col], "books")
if ratings is not None:
    check_cols(ratings, [user_col, rating_item_col, rating_col, timestamp_col], "ratings")
print("Column check finished (no output above the line = all good).")

## 7. Clean the movie dataset

Simple, conservative cleaning. We remove exact duplicates and rows that are unusable
(no ID or no title), fix types, and tidy the text fields. We deliberately avoid
aggressive cleaning that could throw away information the models might need.

In [ ]:
movies_before = len(movies)

# 1. exact duplicate rows
movies = movies.drop_duplicates()

# 2. rows missing the item id or title are unusable for recommendation
movies = movies.dropna(subset=[movie_id_col, movie_title_col])

print(f"Movies: {movies_before} -> {len(movies)} rows "
      f"({movies_before - len(movies)} removed)")

In [ ]:
# 3. fix types and tidy text
movies[movie_id_col] = movies[movie_id_col].astype(str).str.strip()
movies[movie_year_col] = pd.to_numeric(movies[movie_year_col], errors="coerce")

text_cols_movies = [movie_title_col, movie_genre_col, movie_desc_col,
                    movie_director_col, movie_cast_col]

# values that really mean "missing"
EMPTY_VALUES = ["", "unknown", "none", "nan", "n/a", "na"]

for col in text_cols_movies:
    if col in movies.columns:
        movies[col] = movies[col].astype(str).str.strip()
        # turn obvious empty strings back into real missing values
        movies[col] = movies[col].where(~movies[col].str.lower().isin(EMPTY_VALUES), np.nan)

movies[[movie_id_col, movie_title_col, movie_year_col]].head()

In [ ]:
# 4. fill missing content fields with "" so text concatenation later is painless
#    (we keep release_year as NaN - filling it with a fake number would be misleading)
for col in [movie_genre_col, movie_desc_col, movie_director_col, movie_cast_col]:
    if col in movies.columns:
        movies[col] = movies[col].fillna("")

print("Remaining missing values in movies:")
print(movies.isna().sum()[movies.isna().sum() > 0])

## 8. Clean the book dataset

Same idea as the movies: duplicates, unusable rows, types, and text tidying.
ISBNs are kept as **strings** because leading zeros matter and they are identifiers,
not numbers.

In [ ]:
books_before = len(books)

books = books.drop_duplicates()
books = books.dropna(subset=[book_id_col, book_title_col])

print(f"Books: {books_before} -> {len(books)} rows "
      f"({books_before - len(books)} removed)")

In [ ]:
# IDs as clean strings (important for ISBNs)
books[book_id_col] = books[book_id_col].astype(str).str.strip()

# publication year to numeric
books[book_year_col] = pd.to_numeric(books[book_year_col], errors="coerce")

text_cols_books = [book_title_col, book_author_col, book_genre_col, book_desc_col]
for col in text_cols_books:
    if col in books.columns:
        books[col] = books[col].astype(str).str.strip()
        books[col] = books[col].where(~books[col].str.lower().isin(EMPTY_VALUES), np.nan)

books[[book_id_col, book_title_col, book_year_col]].head()

In [ ]:
# look at suspicious publication years BEFORE deciding what to do about them
year_ok = books[book_year_col].between(1400, 2026)
weird_years = books.loc[~year_ok & books[book_year_col].notna(), book_year_col]

print(f"Books with unrealistic publication years: {len(weird_years)} "
      f"({len(weird_years) / len(books) * 100:.2f}% of books)")
print(weird_years.value_counts().head(10))

# we don't delete these rows - the book itself is still valid,
# we just treat the bad year as missing
books.loc[~year_ok, book_year_col] = np.nan

In [ ]:
# fill missing text content with "" like we did for movies
for col in [book_author_col, book_genre_col, book_desc_col]:
    if col in books.columns:
        books[col] = books[col].fillna("")

print("Remaining missing values in books:")
print(books.isna().sum()[books.isna().sum() > 0])

## 9. Clean the interaction / rating data

Interactions are the core input for collaborative filtering, and how sparse they are
is exactly what makes cold start hard. We clean carefully and **look before we delete**.

In [ ]:
if ratings is not None:
    ratings_before = len(ratings)

    # rows missing user or item id are useless
    ratings = ratings.dropna(subset=[user_col, rating_item_col])

    # IDs as strings so they match the item tables later
    ratings[user_col] = ratings[user_col].astype(str).str.strip()
    ratings[rating_item_col] = ratings[rating_item_col].astype(str).str.strip()

    # ratings to numeric
    ratings[rating_col] = pd.to_numeric(ratings[rating_col], errors="coerce")
    ratings = ratings.dropna(subset=[rating_col])

    # duplicate user-item-rating records
    ratings = ratings.drop_duplicates(subset=[user_col, rating_item_col, rating_col])

    print(f"Ratings: {ratings_before} -> {len(ratings)} rows "
          f"({ratings_before - len(ratings)} removed)")

In [ ]:
if ratings is not None:
    print("Rating value counts:")
    print(ratings[rating_col].value_counts().sort_index())
    print("\nMin rating:", ratings[rating_col].min())
    print("Max rating:", ratings[rating_col].max())

    # check for ratings outside the expected scale - update these bounds for your data
    expected_min, expected_max = 1, 5
    out_of_scale = ratings[(ratings[rating_col] < expected_min) |
                           (ratings[rating_col] > expected_max)]
    print(f"\nRatings outside [{expected_min}, {expected_max}]: {len(out_of_scale)}")
    if len(out_of_scale) > 0:
        display(out_of_scale.head())
        # decide manually whether to drop these - we do NOT drop them automatically

In [ ]:
if ratings is not None and timestamp_col is not None and timestamp_col in ratings.columns:
    # many datasets store unix seconds; pd.to_datetime handles both cases with errors="coerce"
    ts = ratings[timestamp_col]
    if pd.api.types.is_numeric_dtype(ts):
        ratings[timestamp_col] = pd.to_datetime(ts, unit="s", errors="coerce")
    else:
        ratings[timestamp_col] = pd.to_datetime(ts, errors="coerce")

    ratings = ratings.sort_values(timestamp_col).reset_index(drop=True)
    print("Timestamp range:", ratings[timestamp_col].min(), "->", ratings[timestamp_col].max())
    HAS_TIMESTAMP = ratings[timestamp_col].notna().mean() > 0.9  # mostly valid timestamps
else:
    HAS_TIMESTAMP = False

print("Using timestamps for splitting:", HAS_TIMESTAMP)

## 10. Basic exploratory analysis

A few simple plots and numbers. The main thing we care about here is **sparsity**:
how many users and items have very few interactions, because those are exactly the
cold-start cases our system needs to handle.

In [ ]:
if ratings is not None:
    plt.figure(figsize=(6, 4))
    ratings[rating_col].hist(bins=20)
    plt.title("Rating distribution")
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.show()

In [ ]:
if ratings is not None:
    interactions_per_user = ratings.groupby(user_col).size()
    interactions_per_item = ratings.groupby(rating_item_col).size()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(interactions_per_user, bins=50)
    axes[0].set_title("Interactions per user")
    axes[0].set_xlabel("Number of interactions")
    axes[0].set_ylabel("Users")

    axes[1].hist(interactions_per_item, bins=50)
    axes[1].set_title("Interactions per item")
    axes[1].set_xlabel("Number of interactions")
    axes[1].set_ylabel("Items")
    plt.tight_layout()
    plt.show()

In [ ]:
if ratings is not None:
    print("Most frequently rated items (raw ids):")
    print(interactions_per_item.sort_values(ascending=False).head(10))

    few_user = (interactions_per_user <= 5).mean() * 100
    few_item = (interactions_per_item <= 5).mean() * 100
    print(f"\nUsers with <= 5 interactions: {few_user:.1f}%")
    print(f"Items with <= 5 interactions: {few_item:.1f}%")

In [ ]:
# how much content information is missing? (matters for content-based fallback)
def pct_empty(series):
    return (series.fillna("").astype(str).str.strip() == "").mean() * 100

print("Movies - % empty content fields:")
for col in [movie_genre_col, movie_desc_col, movie_director_col]:
    if col in movies.columns:
        print(f"  {col}: {pct_empty(movies[col]):.1f}%")

print("Books - % empty content fields:")
for col in [book_genre_col, book_desc_col, book_author_col]:
    if col in books.columns:
        print(f"  {col}: {pct_empty(books[col]):.1f}%")

**What this tells us about cold start (edit after running):**

- If a large share of users/items have ≤5 interactions, the "long tail" is big and
  pure collaborative filtering will struggle for most of the catalog.
- The most-rated items show how concentrated the interactions are on popular items.
- Missing content fields matter because content is our fallback for cold-start items —
  an item with no description *and* no interactions is the hardest possible case.


## 11. Create a unified item table

We combine movies and books into one item table with standardized columns.
IDs get a `movie_` / `book_` prefix so they can never collide. Columns that only
exist in one dataset are simply created as blanks in the other before concatenating.

In [ ]:
# standardized movie items
movie_items = pd.DataFrame({
    "item_id": "movie_" + movies[movie_id_col].astype(str),
    "item_type": "movie",
    "title": movies[movie_title_col],
    "creator": movies[movie_director_col] if movie_director_col in movies.columns else "",
    "genre": movies[movie_genre_col] if movie_genre_col in movies.columns else "",
    "description": movies[movie_desc_col] if movie_desc_col in movies.columns else "",
    "release_year": movies[movie_year_col] if movie_year_col in movies.columns else np.nan,
})

# standardized book items
book_items = pd.DataFrame({
    "item_id": "book_" + books[book_id_col].astype(str),
    "item_type": "book",
    "title": books[book_title_col],
    "creator": books[book_author_col] if book_author_col in books.columns else "",
    "genre": books[book_genre_col] if book_genre_col in books.columns else "",
    "description": books[book_desc_col] if book_desc_col in books.columns else "",
    "release_year": books[book_year_col] if book_year_col in books.columns else np.nan,
})

items = pd.concat([movie_items, book_items], ignore_index=True)
items = items.drop_duplicates(subset=["item_id"])

print("Unified item table:", items.shape)
print(items["item_type"].value_counts())
items.head(8)

## 12. Create a combined text feature

We build one `content_text` column per item by joining the title, type, genre,
creator (author/director), and description.

This text is the **content representation** of each item. Later, a generative model,
an embedding model, or a simple TF-IDF content-based recommender can use it to
recommend brand-new items that have zero interactions — which is the whole point
of the new-item cold-start setup. We are **not** generating embeddings here.

In [ ]:
items["content_text"] = (
    items["title"].fillna("") + " " +
    items["item_type"].fillna("") + " " +
    items["genre"].fillna("") + " " +
    items["creator"].fillna("") + " " +
    items["description"].fillna("")
)

# tidy up: collapse repeated spaces and lowercase
items["content_text"] = (
    items["content_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

items[["item_id", "content_text"]].head()

## 13. Prepare interaction data for modeling

The ratings file uses raw IDs, so we map them onto the prefixed IDs from the unified
item table. Because a raw ID could belong to either dataset, we check membership in
the movie IDs first, then the book IDs.

We do **not** silently drop interactions that don't match an item — we count them first.

In [ ]:
if ratings is not None:
    movie_ids = set(movies[movie_id_col].astype(str))
    book_ids = set(books[book_id_col].astype(str))

    def to_prefixed_id(raw_id):
        if raw_id in movie_ids:
            return "movie_" + raw_id
        if raw_id in book_ids:
            return "book_" + raw_id
        return None  # no matching item

    interactions = ratings.copy()
    interactions["item_id"] = interactions[rating_item_col].map(to_prefixed_id)

    unmatched = interactions["item_id"].isna().sum()
    print(f"Interactions that match no item: {unmatched} "
          f"({unmatched / len(interactions) * 100:.2f}%)")

    # NOTE: if your ratings file already has an item_type column or separate
    # movie/book rating files, prefix the ids directly instead of using the lookup above.

In [ ]:
if ratings is not None:
    # keep only matched interactions and add item_type
    interactions = interactions.dropna(subset=["item_id"]).copy()
    interactions = interactions.merge(items[["item_id", "item_type"]], on="item_id", how="left")

    keep_cols = [user_col, "item_id", rating_col, "item_type"]
    if HAS_TIMESTAMP:
        keep_cols.insert(3, timestamp_col)
    interactions = interactions[keep_cols].rename(
        columns={user_col: "user_id", rating_col: "rating", timestamp_col: "timestamp"}
        if HAS_TIMESTAMP else {user_col: "user_id", rating_col: "rating"})

    print("Clean interaction table:", interactions.shape)
    interactions.head()

## 14. Define the cold-start evaluation strategy

A completely random split is **not enough** for cold-start evaluation: with a random
split, the same users and items usually appear in both training and test, so the test
set never actually contains a "new" user or item. To evaluate cold start honestly,
we need test sets where the users/items are guaranteed to be unseen during training.

We build four evaluation datasets:

| Split | Purpose |
|---|---|
| A. Standard train/val/test | general model development |
| B. New-user cold-start test | users completely absent from training |
| C. New-item cold-start test | items completely absent from training |
| D. Sparse-user test (optional) | users with only 1–5 interactions (partial cold start) |


### 14A. Standard interaction split (70 / 15 / 15)

If timestamps exist we split chronologically (train on the past, test on the future),
which is more realistic. Otherwise we use a reproducible random split.

Note: the cold-start users/items are removed **first** (sections 14B/14C select them),
so we do this split after carving those out. To keep the notebook readable we select
the cold-start users and items now, then split the remainder.

In [ ]:
if ratings is not None:
    # --- select cold-start USERS (14B) ---
    COLD_USER_FRAC = 0.10   # easy to change

    user_counts = interactions.groupby("user_id").size()
    single_interaction_users = (user_counts == 1).sum()
    print(f"Users with only 1 interaction: {single_interaction_users} "
          f"({single_interaction_users / len(user_counts) * 100:.1f}% of users)")

    # users with >= 2 interactions are 'eligible' so the cold-start eval is stable
    eligible_users = user_counts[user_counts >= 2].index.to_series()
    cold_users = eligible_users.sample(frac=COLD_USER_FRAC, random_state=SEED)
    cold_users = set(cold_users)
    print(f"Selected {len(cold_users)} cold-start users")

In [ ]:
if ratings is not None:
    # --- select cold-start ITEMS (14C), sampled separately for movies and books ---
    COLD_ITEM_FRAC = 0.10   # easy to change

    item_counts = interactions.groupby("item_id").size()
    eligible_items = item_counts[item_counts >= 2].index

    eligible_movie_items = pd.Series([i for i in eligible_items if i.startswith("movie_")])
    eligible_book_items = pd.Series([i for i in eligible_items if i.startswith("book_")])

    cold_items = set(eligible_movie_items.sample(frac=COLD_ITEM_FRAC, random_state=SEED)) | \
                 set(eligible_book_items.sample(frac=COLD_ITEM_FRAC, random_state=SEED))
    print(f"Selected {len(cold_items)} cold-start items "
          f"({sum(i.startswith('movie_') for i in cold_items)} movies, "
          f"{sum(i.startswith('book_') for i in cold_items)} books)")

In [ ]:
if ratings is not None:
    # --- carve out the cold-start test sets ---
    is_cold_user = interactions["user_id"].isin(cold_users)
    is_cold_item = interactions["item_id"].isin(cold_items)

    cold_start_user_test = interactions[is_cold_user & ~is_cold_item].copy()
    cold_start_item_test = interactions[is_cold_item].copy()

    # everything else goes into the standard split
    remaining = interactions[~is_cold_user & ~is_cold_item].copy()

    print("Cold-start user test:", cold_start_user_test.shape)
    print("Cold-start item test:", cold_start_item_test.shape)
    print("Remaining for standard split:", remaining.shape)

In [ ]:
if ratings is not None:
    # --- 70 / 15 / 15 standard split ---
    if HAS_TIMESTAMP:
        # chronological: earliest 70% train, next 15% validation, last 15% test
        remaining = remaining.sort_values("timestamp").reset_index(drop=True)
        n = len(remaining)
        train_end = int(n * 0.70)
        val_end = int(n * 0.85)
        train_interactions = remaining.iloc[:train_end].copy()
        validation_interactions = remaining.iloc[train_end:val_end].copy()
        test_interactions = remaining.iloc[val_end:].copy()
        print("Used a CHRONOLOGICAL split (timestamps available).")
    else:
        # reproducible random split
        shuffled = remaining.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n = len(shuffled)
        train_end = int(n * 0.70)
        val_end = int(n * 0.85)
        train_interactions = shuffled.iloc[:train_end].copy()
        validation_interactions = shuffled.iloc[train_end:val_end].copy()
        test_interactions = shuffled.iloc[val_end:].copy()
        print("Used a RANDOM split (no usable timestamps).")

    print("Train:", train_interactions.shape)
    print("Validation:", validation_interactions.shape)
    print("Test:", test_interactions.shape)

### 14D. Optional sparse-user evaluation group

Users with only **1–5 interactions in the training set** are a *partial* cold-start
scenario: the model has seen them, but barely. This is different from the completely
unseen users in 14B, and it's useful for measuring how quickly the system improves
as a user provides their first few ratings.

In [ ]:
if ratings is not None:
    train_user_counts = train_interactions.groupby("user_id").size()
    sparse_users = set(train_user_counts[train_user_counts.between(1, 5)].index)

    # their test-time interactions come from the standard test set
    sparse_user_test = test_interactions[test_interactions["user_id"].isin(sparse_users)].copy()

    print(f"Sparse users (1-5 training interactions): {len(sparse_users)}")
    print("Sparse-user test set:", sparse_user_test.shape)

## 15. Prevent data leakage

Simple, visible checks that the splits actually do what we claim:

- cold-start users never appear in training
- cold-start items never appear in training
- no identical rows shared between training and validation/test
- cold-start items still have their content metadata in the item table
  (that's what a content-based / generative model will use)

Also worth stating: all cleaning decisions above (dropping duplicates, fixing years,
filling blanks) used only per-row information or the full raw data — nothing was
tuned by peeking at test-set performance, so preprocessing itself does not leak.

In [ ]:
if ratings is not None:
    train_users = set(train_interactions["user_id"])
    train_items = set(train_interactions["item_id"])

    # 1. cold-start users absent from training
    assert len(set(cold_start_user_test["user_id"]) & train_users) == 0, \
        "LEAK: cold-start users found in training!"

    # 2. cold-start items absent from training
    assert len(set(cold_start_item_test["item_id"]) & train_items) == 0, \
        "LEAK: cold-start items found in training!"

    # 3. no duplicated user-item pairs between train and val/test
    train_pairs = set(zip(train_interactions["user_id"], train_interactions["item_id"]))
    val_pairs = set(zip(validation_interactions["user_id"], validation_interactions["item_id"]))
    test_pairs = set(zip(test_interactions["user_id"], test_interactions["item_id"]))
    print("Train/val overlapping user-item pairs:", len(train_pairs & val_pairs))
    print("Train/test overlapping user-item pairs:", len(train_pairs & test_pairs))

    # 4. content metadata still available for cold-start items
    missing_meta = cold_items - set(items["item_id"])
    assert len(missing_meta) == 0, "Some cold-start items are missing from the item table!"

    print("\nAll leakage checks passed.")

## 16. Summarize the final splits

In [ ]:
if ratings is not None:
    def split_summary(df, name):
        row = {
            "dataset": name,
            "rows": len(df),
            "unique_users": df["user_id"].nunique(),
            "unique_items": df["item_id"].nunique(),
            "pct_of_all_interactions": round(len(df) / len(interactions) * 100, 2),
            "movies": (df["item_type"] == "movie").sum(),
            "books": (df["item_type"] == "book").sum(),
        }
        return row

    split_table = pd.DataFrame([
        split_summary(train_interactions, "train"),
        split_summary(validation_interactions, "validation"),
        split_summary(test_interactions, "test"),
        split_summary(cold_start_user_test, "cold_start_user_test"),
        split_summary(cold_start_item_test, "cold_start_item_test"),
        split_summary(sparse_user_test, "sparse_user_test"),
    ])
    display(split_table)

## 17. Save the cleaned files

Everything the modeling team needs, saved to `processed_data/`.

In [ ]:
out_dir = "processed_data"
os.makedirs(out_dir, exist_ok=True)

items.to_csv(os.path.join(out_dir, "processed_items.csv"), index=False)

if ratings is not None:
    interactions.to_csv(os.path.join(out_dir, "processed_interactions.csv"), index=False)
    train_interactions.to_csv(os.path.join(out_dir, "train_interactions.csv"), index=False)
    validation_interactions.to_csv(os.path.join(out_dir, "validation_interactions.csv"), index=False)
    test_interactions.to_csv(os.path.join(out_dir, "test_interactions.csv"), index=False)
    cold_start_user_test.to_csv(os.path.join(out_dir, "cold_start_user_test.csv"), index=False)
    cold_start_item_test.to_csv(os.path.join(out_dir, "cold_start_item_test.csv"), index=False)
    sparse_user_test.to_csv(os.path.join(out_dir, "sparse_user_test.csv"), index=False)

print("Saved files:")
for f in sorted(os.listdir(out_dir)):
    print(" -", f)

## 18. Final summary

**What this notebook did:**

- **Cleaning:** removed exact duplicates and rows missing an ID or title from the movie
  and book data; standardized text fields; converted years and ratings to numeric;
  treated placeholder strings ("unknown", "n/a", etc.) as missing; flagged (but did not
  automatically delete) unrealistic publication years and out-of-scale ratings.
- **Standardization:** both catalogs were mapped to one schema
  (`item_id`, `item_type`, `title`, `creator`, `genre`, `description`, `release_year`)
  with `movie_` / `book_` ID prefixes to prevent collisions.
- **Content text:** a `content_text` column combines title, type, genre, creator and
  description per item — this is the input for content-based / generative approaches
  to new-item cold start.
- **Splits:** interactions were divided into a 70/15/15 train/validation/test split
  (chronological when timestamps exist, otherwise seeded random), plus a **new-user**
  cold-start test set (~10% of eligible users, fully removed from training), a
  **new-item** cold-start test set (~10% of eligible movies and books, sampled
  separately per type), and a **sparse-user** test group (users with 1–5 training
  interactions).
- **Leakage checks:** assertions confirm cold-start users/items are absent from
  training and that cold-start items keep their metadata.
- **Outputs:** all tables saved as CSVs in `processed_data/` for the modeling team.

**Assumptions made (confirm with the team):**

- Movie and book IDs in the ratings file can be matched by simple membership lookup
  (a raw ID belongs to exactly one catalog).
- The rating scale is 1–5 (change `expected_min` / `expected_max` if not).
- Users/items need ≥ 2 interactions to be eligible for the cold-start samples.
- Publication years outside 1400–2026 are data errors.

---

### Variables / columns to confirm after loading the real datasets

| Variable | Current placeholder | Check against |
|---|---|---|
| `movie_path`, `book_path`, `ratings_path` | `data/*.csv` | actual file locations |
| `movie_id_col` | `show_id` | movie file columns |
| `movie_genre_col` | `listed_in` | movie file columns |
| `movie_director_col`, `movie_cast_col`, `movie_desc_col`, `movie_year_col` | see §6 | movie file columns |
| `book_id_col` | `isbn` | book file columns |
| `book_author_col`, `book_genre_col`, `book_desc_col`, `book_year_col` | see §6 | book file columns |
| `user_col`, `rating_item_col`, `rating_col`, `timestamp_col` | see §6 | ratings file columns |
| `expected_min`, `expected_max` | 1, 5 | real rating scale |
| `COLD_USER_FRAC`, `COLD_ITEM_FRAC` | 0.10 | team decision |
